In [6]:

import numpy as np


def sig(s):
    """Función de activación Sigmoide."""
    return 1.0 / (1.0 + np.exp(-s))


def dSig(s):
    """Derivada de la función sigmoide en forma matricial diagonal[cite: 21]."""
    df = sig(s.ravel()) * (1.0 - sig(s.ravel()))
    return np.diag(df)


def error(A, Yd):
    """Cálculo del error entre la salida deseada (Yd) y la producida (A)[cite: 22]."""
    return Yd - A

In [7]:

class RedNeuronal2CapasOcultas:

    def __init__(self, nn_in=2, nn_h1=3, nn_h2=3, nn_out=1, seed=42):
        """Inicialización de pesos y sesgos para 2 capas ocultas[cite: 21]."""
        np.random.seed(seed)
        # Capa 1 (Entrada -> Capa Oculta 1)
        self.W1 = np.random.uniform(-1, 1, (nn_h1, nn_in))
        self.b1 = np.random.uniform(-1, 1, (nn_h1, 1))

        # Capa 2 (Capa Oculta 1 -> Capa Oculta 2)
        self.W2 = np.random.uniform(-1, 1, (nn_h2, nn_h1))
        self.b2 = np.random.uniform(-1, 1, (nn_h2, 1))

        # Capa 3 (Capa Oculta 2 -> Capa de Salida)
        self.W3 = np.random.uniform(-1, 1, (nn_out, nn_h2))
        self.b3 = np.random.uniform(-1, 1, (nn_out, 1))

    def propaga(self, Xo):
        """Propagación hacia adelante a través de 3 capas (2 ocultas + 1 salida)[cite: 21]."""
        Z1 = np.dot(self.W1, Xo) + self.b1
        A1 = sig(Z1)

        Z2 = np.dot(self.W2, A1) + self.b2
        A2 = sig(Z2)

        Z3 = np.dot(self.W3, A2) + self.b3
        A3 = sig(Z3)

        return A3, Z3, A2, Z2, A1, Z1

    def backpropagation(self, X, Yr, A3, Z3, A2, Z2, A1, Z1):
        """Retropropagación ajustando los errores desde la capa 3 hacia la capa 1[cite: 19, 22]."""
        # Error y gradientes Capa de Salida (Capa 3)
        delta3 = np.dot(dSig(Z3), error(A3, Yr))
        dEdW3 = -np.dot(delta3, A2.T)
        dEdb3 = -delta3

        # Gradientes Capa Oculta 2 (Capa 2)
        delta2 = np.dot(np.dot(dSig(Z2), self.W3.T), delta3)
        dEdW2 = -np.dot(delta2, A1.T)
        dEdb2 = -delta2

        # Gradientes Capa Oculta 1 (Capa 1)
        delta1 = np.dot(np.dot(dSig(Z1), self.W2.T), delta2)
        dEdW1 = -np.dot(delta1, X.T)
        dEdb1 = -delta1

        return dEdW3, dEdb3, dEdW2, dEdb2, dEdW1, dEdb1

    def entrenar(self, X_data, Y_data, epocas=40000, eta=0.2):
        """Bucle de entrenamiento por descenso de gradiente[cite: 18, 23]."""
        for epoca in range(epocas):
            for i in range(len(X_data)):
                X = X_data[i].reshape(-1, 1)
                Yr = Y_data[i].reshape(-1, 1)

                # Forward
                A3, Z3, A2, Z2, A1, Z1 = self.propaga(X)

                # Backward
                dEdW3, dEdb3, dEdW2, dEdb2, dEdW1, dEdb1 = self.backpropagation(
                    X, Yr, A3, Z3, A2, Z2, A1, Z1
                )

                # Actualización de pesos y sesgos
                self.W3 -= eta * dEdW3
                self.b3 -= eta * dEdb3
                self.W2 -= eta * dEdW2
                self.b2 -= eta * dEdb2
                self.W1 -= eta * dEdW1
                self.b1 -= eta * dEdb1

    def probar(self, X_data, Y_data):
        """Evalúa los resultados finales producidos por la red."""
        print(f"{'Entrada':<12} | {'Esperado':<10} | {'Predicción (A3)':<15} | {'Clase'}")
        print("-" * 55)
        for i in range(len(X_data)):
            X = X_data[i].reshape(-1, 1)
            A3, _, _, _, _, _ = self.propaga(X)
            val_pred = A3.ravel()[0]
            clase = 1 if val_pred >= 0.5 else 0
            print(
                f"{str(X_data[i]):<12} | {Y_data[i][0]:<10} | {val_pred:<15.4f} | {clase}"
            )

In [8]:

# Datos de entrada y rótulos esperados para la compuerta NAND
X_inputs = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])

# Tabla de verdad NAND: Salida 0 solo cuando ambas entradas son 1
Y_nand = np.array([[1], [1], [1], [0]])

print("==========================================")
print("  RED NEURONAL 1: COMPUERTA NAND")
print("==========================================")

# Instanciar y entrenar la red para NAND
red_nand = RedNeuronal2CapasOcultas(
    nn_in=2, nn_h1=3, nn_h2=3, nn_out=1, seed=42
)
red_nand.entrenar(X_inputs, Y_nand, epocas=30000, eta=0.2)
red_nand.probar(X_inputs, Y_nand)

  RED NEURONAL 1: COMPUERTA NAND
Entrada      | Esperado   | Predicción (A3) | Clase
-------------------------------------------------------
[0 0]        | 1          | 0.9990          | 1
[0 1]        | 1          | 0.9940          | 1
[1 0]        | 1          | 0.9938          | 1
[1 1]        | 0          | 0.0093          | 0


In [9]:

# Tabla de verdad XOR: Salida 1 solo cuando las entradas son diferentes
Y_xor = np.array([[0], [1], [1], [0]])

print("\n==========================================")
print("  RED NEURONAL 2: COMPUERTA XOR")
print("==========================================")

# Instanciar y entrenar la red para XOR
red_xor = RedNeuronal2CapasOcultas(
    nn_in=2, nn_h1=3, nn_h2=3, nn_out=1, seed=42
)
red_xor.entrenar(X_inputs, Y_xor, epocas=50000, eta=0.2)
red_xor.probar(X_inputs, Y_xor)


  RED NEURONAL 2: COMPUERTA XOR
Entrada      | Esperado   | Predicción (A3) | Clase
-------------------------------------------------------
[0 0]        | 0          | 0.0100          | 0
[0 1]        | 1          | 0.9904          | 1
[1 0]        | 1          | 0.9904          | 1
[1 1]        | 0          | 0.0097          | 0
